# Stage 2 Notebook 32 - Exp2AA Mask aux + higher resolution training

**Why this exists.** Mask supervision (Exp2W) lifted decoded_f1 from 0.026 to 0.043 (+65%). But matched_iou stayed at 0.143, suggesting the geometry quality is still bottlenecked. One plausible cause: at 384x640 input, lane lines are 1-2 pixels wide -- the backbone has very thin features to localize. CLRKDNet uses 590x1640 on CULane.

Exp2AA scales up:
- `image_size: 384x640 -> 480x800` (1.5x more pixels; 384k vs 246k pixels per image)
- `aux_mask_size: 72x128 -> 90x160` (matches new aspect)
- `batch_size: 8 -> 4` (memory budget at higher res)
- `end_epoch: 10 -> 15` (more training time pays off with stable lambda)
- `lambda_mode: fixed` (Exp2Y's stability fix bundled in)

Independent of Exp2Y/Z: those test the joint-loss-balancing fix at base resolution; this tests whether resolution itself is the bottleneck.

Reference: CLRNet/CLRKDNet train at 590x1640 on CULane; LaneATT and CondLaneNet use 800x288 minimum.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 15-epoch short run.
3. **Note**: this run will be slower per step due to higher resolution. Estimate ~75-90s per epoch (vs ~57s at 384x640).
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [3]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [4]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp27_rmt_gca_mask_high_resolution_joint_smoke.log
Traceback (most recent call last):
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 76, in <module>
    main()
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 72, in main
    run_one(Path(item))
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 53, in run_one
    lane_loss, lane_comp = lane_loss_fn(out['lane'], lane_target)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)


CalledProcessError: Command '['/usr/bin/python3', '-u', 'stage2/scripts/smoke_test_joint_models.py', 'stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp27_rmt_gca_mask_high_resolution_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 2
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short15'
    EPOCHS = 15
    BATCH_SIZE = 4
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in Exp2AA training

Pass criteria at epoch 15:
- **`val/matched_line_iou >= 0.20`**: higher resolution should give thinner-line localization. If it stays at 0.14, resolution isn't the issue.
- **`val/lane/decoded_f1 >= 0.08`**: should beat Exp2W (0.043) by >2x.
- **`val/lane/decoded_oracle_f1 >= 0.15`**: oracle ranking ceiling rises with better geometry.

Failure signals:
- Geometry stays at ~0.14 despite 1.5x more pixels: the bottleneck isn't resolution; pivot to KD from teacher or longer training.
- Memory OOM in debug: bump batch_size down to 2.